# Pose Estimation Model Training in Google Colab

## 1. Setup Environment

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.functional as TF
from torchvision import transforms
import numpy as np
import os, json, random
import matplotlib.pyplot as plt

## 2. Dataset Definition

In [ ]:
import random
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
from torchvision import transforms

FLIP_INDICES = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15, 18, 17, 20, 19, 22, 21, 24, 23, 26, 25, 28, 27, 30, 29, 32, 31]


# This function will be called by our Dataset to create the target heatmaps
def generate_heatmaps(keypoints, output_res, sigma=2):
    """
    Generates 2D Gaussian heatmaps for keypoints.

    Args:
        keypoints (np.array): A (num_keypoints, 2) array of (x, y) coordinates.
        output_res (tuple): The (height, width) of the output heatmaps.
        sigma (float): The standard deviation of the Gaussian kernel.

    Returns:
        torch.Tensor: A (num_keypoints, height, width) tensor of heatmaps.
    """
    heatmaps = np.zeros((keypoints.shape[0], output_res[0], output_res[1]), dtype=np.float32)
    for i, (x, y) in enumerate(keypoints):
        # Ensure keypoints are within bounds
        if x < 0 or x >= output_res[1] or y < 0 or y >= output_res[0]:
            continue

        # Create a grid of coordinates
        xx, yy = np.meshgrid(np.arange(output_res[1]), np.arange(output_res[0]))

        # Generate the Gaussian peak
        heatmap = np.exp(-((xx - x)**2 + (yy - y)**2) / (2 * sigma**2))
        heatmaps[i] = heatmap

    return torch.from_numpy(heatmaps)


class PoseDataset(Dataset):
    def __init__(self, data_dir, num_keypoints=33, output_res=(240, 240), augment=False):
        self.data_dir = data_dir
        self.depth_dir = os.path.join(data_dir, 'depth')
        self.confidence_dir = os.path.join(data_dir, 'confidence')
        self.pose_dir = os.path.join(data_dir, 'pose')

        self.file_list = [f.split('.')[0] for f in os.listdir(self.depth_dir)]
        self.num_keypoints = num_keypoints
        self.output_res = output_res
        self.augment = augment

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        filename = self.file_list[idx]

        # --- Load Sensor Data ---
        depth_path = os.path.join(self.depth_dir, f"{filename}.npy")
        confidence_path = os.path.join(self.confidence_dir, f"{filename}.npy")

        depth_map = np.load(depth_path)
        confidence_map = np.load(confidence_path)

        # --- Load and Process Labels ---
        pose_path = os.path.join(self.pose_dir, f"{filename}.json")

        # NEW: Open and load the JSON data
        with open(pose_path, 'r') as f:
            pose_data = json.load(f)

        # NEW: Extract the 'transformed_points' list and convert to a numpy array
        # This gives us our (33, 2) array of (x, y) coordinates.
        keypoints_2d = np.array(pose_data['transformed_points'])


        # --- Pre-process Input (This part is the same) ---
        # Normalize
        depth_map = depth_map / 4000.0
        confidence_map = confidence_map / 255.0

        # Stack depth and confidence
        input_tensor = torch.from_numpy(np.stack([depth_map, confidence_map], axis=0)).float()


        if self.augment:
            # 1. Random Horizontal Flip (50% chance)
            if random.random() > 0.5:
                # Flip the image tensor
                input_tensor = TF.hflip(input_tensor)
                # Flip the keypoint x-coordinates
                img_width = input_tensor.shape[2]
                keypoints_2d[:, 0] = img_width - 1 - keypoints_2d[:, 0]
                # Swap the left/right keypoint indices
                keypoints_2d = keypoints_2d[FLIP_INDICES]

            # 2. Random Rotation
            angle = (random.random() - 0.5) * 2 * 15 # Random angle between -15 and +15 degrees
            # Rotate image
            input_tensor = TF.rotate(input_tensor, angle)
            # Rotate keypoints around the image center

            # Get image center, explicitly creating a float32 tensor
            center = torch.tensor([input_tensor.shape[2] / 2, input_tensor.shape[1] / 2], dtype=torch.float32)

            # Create rotation matrix, explicitly creating a float32 tensor
            rot_mat = torch.tensor([
                [np.cos(np.radians(-angle)), -np.sin(np.radians(-angle))],
                [np.sin(np.radians(-angle)), np.cos(np.radians(-angle))]
            ], dtype=torch.float32)

            # Now all tensors in the operation below are torch.float32
            keypoints_tensor = torch.from_numpy(keypoints_2d).float() - center
            keypoints_tensor = torch.matmul(keypoints_tensor, rot_mat) + center
            keypoints_2d = keypoints_tensor.numpy()




        # Pad to square
        _, h, w = input_tensor.shape
        pad_left = (self.output_res[1] - w) // 2
        pad_right = self.output_res[1] - w - pad_left
        pad_top = (self.output_res[0] - h) // 2
        pad_bottom = self.output_res[0] - h - pad_top
        padding = (pad_left, pad_top, pad_right, pad_bottom)
        input_tensor = transforms.functional.pad(input_tensor, padding)

        # --- Prepare Labels (This part is the same logic) ---
        # Account for padding in the keypoint coordinates
        keypoints_2d[:, 0] += pad_left
        keypoints_2d[:, 1] += pad_top

        # Generate target heatmaps
        target_heatmaps = generate_heatmaps(keypoints_2d, self.output_res, sigma=5)

        return input_tensor, target_heatmaps

## 3. Model Definition (PoseUNet)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)
    
class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    """Upscaling then double conv, now with ConvTranspose2d"""
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # Use a learnable upsampling layer
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)

        # The subsequent convolution now takes the concatenated channels
        # We are using the standard DoubleConv here again for more capacity
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Pad x1 to the size of x2
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = nn.functional.pad(x1, [diffX // 2, diffX - diffX // 2,
                                    diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class PoseUNet(nn.Module):
    def __init__(self, n_channels, n_keypoints):
        super(PoseUNet, self).__init__()
        self.n_channels = n_channels
        self.n_keypoints = n_keypoints

        # Encoder Path
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        # At the bottom of the U
        self.down4 = Down(512, 1024)

        # Decoder Path (Corrected Channel Numbers)
        # The first Up block (`up1`) must accept the output of `down4` (1024 channels)
        self.up1 = Up(1024, 512)
        # `up2` accepts the output of `up1` (512 channels) concatenated with `down3` (512 channels),
        # so its `DoubleConv` will see 1024 channels. The Up class handles this.
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)
        self.outc = OutConv(64, n_keypoints)

    def forward(self, x):
        # This part remains the same and is logically correct
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

## 4. Training Configuration

In [ ]:
# --- Hyperparameters ---
LEARNING_RATE = 1e-4
BATCH_SIZE = 32
EPOCHS = 150
# IMPORTANT: Update this path to point to your dataset in Google Drive
DATA_DIR = "data/"
NUM_KEYPOINTS = 33
INPUT_CHANNELS = 2
IMAGE_RESOLUTION = (240, 240)
VAL_SPLIT = 0.2
RANDOM_SEED = 630

# --- New Hyperparameters for Advanced Training ---
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.1
EARLY_STOP_PATIENCE = 7

# --- Device Configuration ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Using device: {device}')

## 5. Training Loop

In [ ]:
# --- Model, Loss, Optimizer ---
model = PoseUNet(n_channels=INPUT_CHANNELS, n_keypoints=NUM_KEYPOINTS).to(device)
# loss_function = nn.MSELoss()
pos_weight = torch.tensor([500.0]).to(device)
loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight) # NEW CODE
print(f"Using loss function: {loss_function.__class__.__name__} with pos_weight={pos_weight.item()}")

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 'min', patience=SCHEDULER_PATIENCE, factor=SCHEDULER_FACTOR,
)

# --- Datasets and DataLoaders ---
full_dataset = PoseDataset(data_dir=DATA_DIR, output_res=IMAGE_RESOLUTION, augment=True)
non_aug_dataset = PoseDataset(data_dir=DATA_DIR, output_res=IMAGE_RESOLUTION, augment=False)

dataset_size = len(full_dataset)
val_size = int(dataset_size * VAL_SPLIT)
train_size = dataset_size - val_size
print(f"Total samples: {dataset_size}, Training on: {train_size}, Validating on: {val_size}")

generator = torch.Generator().manual_seed(RANDOM_SEED)
train_indices, val_indices = random_split(range(len(full_dataset)), [train_size, val_size], generator=generator)

train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset = torch.utils.data.Subset(non_aug_dataset, val_indices)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("DataLoaders created. Training set will be augmented, validation set will not.")

# --- Training State Variables ---
best_val_loss = float('inf')
epochs_no_improve = 0
# IMPORTANT: Update this path to your Google Drive
MODEL_SAVE_PATH = "/content/drive/MyDrive/pose_unet_bce_best_model.pth"
# FINAL_MODEL_PATH = "/content/drive/MyDrive/pose_unet_bce_model.pth"

# --- Training Loop ---
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    for data, targets in train_loader:
        data, targets = data.to(device), targets.to(device)
        optimizer.zero_grad()
        predictions = model(data)
        loss = loss_function(predictions, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)

    # --- Validation Loop ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for data, targets in val_loader:
            data, targets = data.to(device), targets.to(device)
            predictions = model(data)
            val_loss = loss_function(predictions, targets)
            total_val_loss += val_loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    # print(f"Epoch {epoch+1}/{EPOCHS} -> Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")
    current_lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Val Loss: {avg_val_loss:.6f} | "
        f"LR: {current_lr:.1e}"
    )

    # --- Scheduler and Early Stopping ---
    scheduler.step(avg_val_loss)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"Validation loss improved. Model saved to {MODEL_SAVE_PATH}")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epochs.")

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break

# --- Save Final Model ---
# torch.save(model.state_dict(), FINAL_MODEL_PATH)
print(f"Finished training.")

## 6. Visualize Ground Truth (Optional)

In [ ]:
def visualize_ground_truth(loader, num_samples):
    """Visualizes ground truth data from a DataLoader."""
    for i, (inputs, targets) in enumerate(loader):
        if i * loader.batch_size >= num_samples:
            break

        inputs = inputs.cpu().numpy()
        targets = targets.cpu().numpy()

        for j in range(inputs.shape[0]):
            sample_idx = i * loader.batch_size + j
            if sample_idx >= num_samples:
                break

            depth_map = inputs[j, 0, :, :]
            confidence_map = inputs[j, 1, :, :]
            summed_heatmaps = np.sum(targets[j], axis=0)

            fig, axs = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle(f'Sample #{sample_idx}', fontsize=16)

            im1 = axs[0].imshow(depth_map, cmap='viridis')
            axs[0].set_title('Padded Depth Map (Input Ch 1)')
            axs[0].axis('off')
            fig.colorbar(im1, ax=axs[0], fraction=0.046, pad=0.04)

            im2 = axs[1].imshow(confidence_map, cmap='magma')
            axs[1].set_title('Padded Confidence Map (Input Ch 2)')
            axs[1].axis('off')
            fig.colorbar(im2, ax=axs[1], fraction=0.046, pad=0.04)

            im3 = axs[2].imshow(summed_heatmaps, cmap='hot')
            axs[2].set_title('Summed Heatmaps (Ground Truth)')
            axs[2].axis('off')
            fig.colorbar(im3, ax=axs[2], fraction=0.046, pad=0.04)

            axs[3].imshow(depth_map, cmap='viridis')
            axs[3].imshow(summed_heatmaps, cmap='hot', alpha=0.6)
            axs[3].set_title('Overlay: Heatmaps on Depth')
            axs[3].axis('off')

            plt.tight_layout()
            plt.show()

vis_loader = DataLoader(dataset=val_dataset, batch_size=4, shuffle=True)
visualize_ground_truth(vis_loader, num_samples=8)

## 7. Visualize Predictions

In [ ]:
# --- Configuration for Visualization ---
# This should point to the model you just saved
VIS_MODEL_PATH = "/content/drive/MyDrive/pose_unet_best_model.pth"
# This points to the same data directory used for training
VIS_DATA_DIR = "/content/drive/MyDrive/your_data_folder/data"
VIS_BATCH_SIZE = 4
NUM_SAMPLES_TO_VISUALIZE = 8

# Use the same validation split parameters to get the correct validation set
# This MUST be the same seed used in your training script!
VIS_RANDOM_SEED = RANDOM_SEED # Use the same seed from training

# --- Model and Data Setup ---
if not os.path.exists(VIS_MODEL_PATH):
    print(f"ERROR: Model file not found at {VIS_MODEL_PATH}.")
    print("Please make sure the path is correct and the model was saved.")
else:
    # 1. Initialize the model architecture
    vis_model = PoseUNet(n_channels=INPUT_CHANNELS, n_keypoints=NUM_KEYPOINTS).to(device)

    # 2. Load the saved weights
    vis_model.load_state_dict(torch.load(VIS_MODEL_PATH, map_location=device))
    print("Model weights loaded successfully for visualization.")

    # 3. Recreate the exact same validation split to get the correct validation data
    vis_full_dataset = PoseDataset(data_dir=VIS_DATA_DIR, output_res=IMAGE_RESOLUTION, augment=False)
    dataset_size = len(vis_full_dataset)
    val_size = int(dataset_size * VAL_SPLIT)
    train_size = dataset_size - val_size
    generator = torch.Generator().manual_seed(VIS_RANDOM_SEED)
    _, vis_val_dataset = random_split(vis_full_dataset, [train_size, val_size], generator=generator)

    # 4. Set up the DataLoader
    vis_loader = DataLoader(dataset=vis_val_dataset, batch_size=VIS_BATCH_SIZE, shuffle=True)
    print(f"Created DataLoader with {len(vis_val_dataset)} validation samples for visualization.")

    # --- NEW HELPER FUNCTION ---
    def get_coords_from_heatmaps(heatmaps_tensor):
        """
        Takes a tensor of heatmaps and returns the (x, y) coordinates of the maximum value in each map.
        Args:
            heatmaps_tensor (torch.Tensor): A tensor of shape (num_keypoints, height, width).
        Returns:
            np.ndarray: An array of shape (num_keypoints, 2) with the (x, y) coordinates.
        """
        num_keypoints, height, width = heatmaps_tensor.shape
        coords = np.zeros((num_keypoints, 2), dtype=np.int32)

        for k in range(num_keypoints):
            heatmap = heatmaps_tensor[k, :, :]
            # Find the flattened index of the max value
            max_index = torch.argmax(heatmap)
            # Convert the flattened index to 2D coordinates
            y = max_index // width
            x = max_index % width
            coords[k] = [x.item(), y.item()]

        return coords

    def visualize_predictions_with_points(model, loader, num_samples, device):
        """
        Visualizes model predictions, including the final calculated keypoints.
        """
        model.eval()
        print("\nDisplaying model predictions with calculated keypoints...")

        with torch.no_grad():
            for i, (inputs, targets) in enumerate(loader):
                if i * loader.batch_size >= num_samples:
                    break

                inputs = inputs.to(device)
                predicted_logits = model(inputs)
                predicted_heatmaps = torch.sigmoid(predicted_logits)

                # Move all data to CPU for plotting and processing
                inputs_np = inputs.cpu().numpy()
                targets_np = targets.cpu().numpy()
                predicted_heatmaps_torch = predicted_heatmaps.cpu() # Keep as tensor for helper func

                for j in range(inputs_np.shape[0]):
                    sample_idx = i * loader.batch_size + j
                    if sample_idx >= num_samples:
                        break

                    depth_map = inputs_np[j, 0, :, :]
                    summed_ground_truth = np.sum(targets_np[j], axis=0)
                    summed_prediction = np.sum(predicted_heatmaps_torch[j].numpy(), axis=0)

                    # --- NEW: Get coordinates from both ground truth and prediction ---
                    gt_coords = get_coords_from_heatmaps(torch.from_numpy(targets_np[j]))
                    pred_coords = get_coords_from_heatmaps(predicted_heatmaps_torch[j])

                    # --- Plotting ---
                    fig, axs = plt.subplots(1, 5, figsize=(25, 5)) # Changed to 5 plots
                    fig.suptitle(f'Validation Sample #{sample_idx}', fontsize=16)

                    # Panels 1-4 are the same as before
                    axs[0].imshow(depth_map, cmap='viridis', aspect='equal')
                    axs[0].set_title('Input Depth Map'); axs[0].axis('off')
                    axs[1].imshow(summed_ground_truth, cmap='hot', aspect='equal')
                    axs[1].set_title('Ground Truth Heatmap'); axs[1].axis('off')
                    axs[2].imshow(summed_prediction, cmap='hot', aspect='equal')
                    axs[2].set_title('Model Prediction Heatmap'); axs[2].axis('off')
                    axs[3].imshow(depth_map, cmap='viridis', aspect='equal')
                    axs[3].imshow(summed_prediction, cmap='hot', alpha=0.6, aspect='equal')
                    axs[3].set_title('Prediction Overlay'); axs[3].axis('off')

                    # --- NEW: 5th Panel for Final Coordinate Comparison ---
                    axs[4].imshow(depth_map, cmap='viridis', aspect='equal')
                    # Plot Ground Truth points in green
                    axs[4].scatter(gt_coords[:, 0], gt_coords[:, 1], s=20, c='lime', marker='o', label='Ground Truth')
                    # Plot Predicted points in red
                    axs[4].scatter(pred_coords[:, 0], pred_coords[:, 1], s=20, c='red', marker='x', label='Prediction')
                    axs[4].set_title('Final Points Overlay')
                    axs[4].axis('off')
                    axs[4].legend()

                    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
                    plt.show()

    # --- Run the visualization ---
    visualize_predictions_with_points(vis_model, vis_loader, num_samples=NUM_SAMPLES_TO_VISUALIZE, device=device)